# ScrollNudge: analysis

This notebook has two parts.

1. **After week 1:** train the mindless-scrolling detector from your check-in answers and export `model.json` for the extension.
2. **After week 3:** compare nudge days with no-nudge days.

Export your data from the extension popup and put the three CSV files in a folder called `data` next to this notebook.

In [ ]:
import glob
import json
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve

DATA = Path('data')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)
FEATURES = ['elapsed_min', 'speed_2m', 'clicks_pm', 'reversals_pm', 'pause_frac', 'hour_sin', 'hour_cos', 'since_last_min']
MINDLESS_MAX = 2

def latest(pattern):
    files = sorted(glob.glob(str(DATA / pattern)))
    if not files:
        raise FileNotFoundError(f'No file matching {pattern} in {DATA}/')
    return files[-1]

w = pd.read_csv(latest('scrollnudge_windows_*.csv'))
e = pd.read_csv(latest('scrollnudge_events_*.csv'))
w['time'] = pd.to_datetime(w['t'], unit='ms')
e['time'] = pd.to_datetime(e['t'], unit='ms')
print(f'{len(w)} windows, {len(e)} events, {w["date"].nunique()} days')
print(w.groupby('condition')['date'].nunique().rename('days'))

## Sessions

A session is continuous time on one site with no gap longer than 3 minutes.

In [ ]:
s = (w.groupby('session_id')
       .agg(date=('date', 'first'), day=('day', 'first'), condition=('condition', 'first'), site=('site', 'first'),
            start=('time', 'min'), end=('time', 'max'), minutes=('dur_s', lambda x: x.sum() / 60))
       .reset_index())
daily = (w.groupby(['date', 'day', 'condition'])
           .agg(minutes=('dur_s', lambda x: x.sum() / 60))
           .join(s.groupby(['date', 'day', 'condition']).agg(sessions=('session_id', 'count'),
                                                              mean_session=('minutes', 'mean'),
                                                              long_sessions=('minutes', lambda x: (x > 20).sum())))
           .reset_index())
print(s.groupby('site')['minutes'].agg(['count', 'sum', 'mean']).round(1).sort_values('sum', ascending=False))
daily.round(1)

## Part 1: train the detector (after week 1)

Each check-in answer is matched to the scrolling features at the moment the question appeared. Answers of 1 or 2 count as mindless.

In [ ]:
shown = e[e['type'] == 'prompt_shown'][['t']].rename(columns={'t': 'token'})
answers = e[e['type'] == 'prompt_answer'][['token', 'value']].copy()
answers['token'] = answers['token'].astype('int64')
answers['rating'] = answers['value'].astype(int)
labels = answers.merge(w, left_on='token', right_on='t', how='inner')
labels['mindless'] = (labels['rating'] <= MINDLESS_MAX).astype(int)
print(f'{len(labels)} answered check-ins, {labels["mindless"].sum()} mindless')
print(labels['rating'].value_counts().sort_index().rename('answers'))

In [ ]:
from sklearn.model_selection import GroupKFold

train = labels[labels['condition'] == 'baseline'] if (labels['condition'] == 'baseline').sum() >= 20 else labels
X, y, days = train[FEATURES].values, train['mindless'].values, train['day'].values
n_pos, n_neg = int(y.sum()), int(len(y) - y.sum())
print(f'Training answers: {len(y)} ({n_pos} mindless, {n_neg} intentional) from {len(set(days))} days')
if n_pos < 5 or n_neg < 5 or len(set(days)) < 3:
    print('Not enough answers of both kinds yet (need at least 5 of each over 3+ days).')
    print('Keep the starter rule for now, keep answering check-ins, and run this cell again.')
else:
    pred = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=min(5, len(set(days)))).split(X, y, days):
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
        if y[tr].min() == y[tr].max():
            pred[te] = y[tr].mean()
            continue
        m = LogisticRegression(C=0.5, max_iter=1000, class_weight='balanced').fit((X[tr] - mu) / sd, y[tr])
        pred[te] = m.predict_proba((X[te] - mu) / sd)[:, 1]
    auc = roc_auc_score(y, pred)
    p, r, t = precision_recall_curve(y, pred)
    f1 = 2 * p * r / (p + r + 1e-9)
    best = int(np.argmax(f1[:-1]))
    threshold = float(t[best])
    print(f'Cross-validated AUC (folds split by day): {auc:.3f}')
    print(f'Chosen threshold: {threshold:.3f} (precision {p[best]:.2f}, recall {r[best]:.2f})')
    if auc < 0.6:
        print('The detector is weak (AUC below 0.6). It will still work, but treat its nudges as close to a timer.')

    mu, sd = X.mean(0), X.std(0) + 1e-9
    final = LogisticRegression(C=0.5, max_iter=1000, class_weight='balanced').fit((X - mu) / sd, y)
    model = {'features': FEATURES, 'mean': mu.tolist(), 'std': sd.tolist(), 'coef': final.coef_[0].tolist(),
             'intercept': float(final.intercept_[0]), 'threshold': threshold, 'cv_auc': float(auc), 'n_labels': int(len(y))}
    Path('model.json').write_text(json.dumps(model, indent=1))
    print('Saved model.json. Load it from the extension popup before day 8.')
    print(pd.Series(final.coef_[0], index=FEATURES).sort_values().round(3).rename('weight (standardized)'))

## Part 2: did the nudges work? (after week 3)

Each experiment day was randomly assigned before the study began, so differences between nudge days and no-nudge days are not caused by which days you chose. The permutation test reshuffles the day labels in every possible way and asks how often a difference this large appears by chance.

In [ ]:
exp = daily[daily['condition'].isin(['nudge', 'control'])].copy()

def exact_perm(values, is_nudge):
    values, is_nudge = np.asarray(values, float), np.asarray(is_nudge, bool)
    obs = values[is_nudge].mean() - values[~is_nudge].mean()
    k, n = is_nudge.sum(), len(values)
    diffs = []
    for idx in combinations(range(n), k):
        mask = np.zeros(n, bool)
        mask[list(idx)] = True
        diffs.append(values[mask].mean() - values[~mask].mean())
    diffs = np.array(diffs)
    return obs, float((np.abs(diffs) >= abs(obs) - 1e-12).mean())

if exp['condition'].nunique() < 2:
    print('The experiment phase has not finished yet.')
else:
    rows = []
    for col, name in [('minutes', 'Scrolling minutes per day'), ('sessions', 'Sessions per day'),
                      ('mean_session', 'Mean session length (min)'), ('long_sessions', 'Sessions over 20 min per day')]:
        v = exp[col].fillna(0)
        obs, p = exact_perm(v, exp['condition'] == 'nudge')
        rows.append({'measure': name, 'nudge days': v[exp['condition'] == 'nudge'].mean(),
                     'no-nudge days': v[exp['condition'] == 'control'].mean(), 'difference': obs, 'p (exact permutation)': p})
    results = pd.DataFrame(rows).round(3)
    display(results)

### Did people (you) stop after a nudge?

On no-nudge days the detector still records the moments it would have nudged ("shadow nudges"). Comparing real nudges with shadow nudges compares the same kind of moment with and without the message.

In [ ]:
trig = e[e['type'].isin(['nudge', 'shadow_nudge'])].merge(s[['session_id', 'end']], on='session_id', how='left')
trig['minutes_left'] = (trig['end'] - trig['time']).dt.total_seconds() / 60
trig['stopped_3min'] = trig['minutes_left'] <= 3
trig = trig[trig['condition'].isin(['nudge', 'control'])]
if trig['type'].nunique() < 2:
    print('Need both nudges and shadow nudges from the experiment phase.')
else:
    summary = trig.groupby('type').agg(moments=('stopped_3min', 'size'), stopped_within_3_min=('stopped_3min', 'mean'),
                                       median_minutes_left=('minutes_left', 'median'))
    display(summary.round(3))
    resp = e[e['type'] == 'nudge_response']['value'].value_counts()
    print('Your answers to nudges:')
    print(resp)
    rng = np.random.default_rng(0)
    a = trig.loc[trig['type'] == 'nudge', 'stopped_3min'].values
    b = trig.loc[trig['type'] == 'shadow_nudge', 'stopped_3min'].values
    obs = a.mean() - b.mean()
    pool = np.concatenate([a, b])
    sims = []
    for _ in range(10000):
        rng.shuffle(pool)
        sims.append(pool[:len(a)].mean() - pool[len(a):].mean())
    print(f'Difference in stopping rate: {obs:+.3f}, permutation p = {(np.abs(sims) >= abs(obs)).mean():.4f}')

In [ ]:
rat = labels[labels['condition'].isin(['nudge', 'control'])]
if rat['condition'].nunique() == 2:
    print('Mean intentionality rating (1 = mindless, 5 = on purpose):')
    print(rat.groupby('condition')['rating'].agg(['count', 'mean']).round(2))

## Figures

In [ ]:
colors = {'baseline': '#8A8F98', 'nudge': '#3A9D5D', 'control': '#2F6DB5', 'after': '#cccccc'}
names = {'baseline': 'Baseline', 'nudge': 'Nudge day', 'control': 'No-nudge day', 'after': 'After study'}
fig, ax = plt.subplots(figsize=(9, 3.5))
d = daily.sort_values('day')
ax.bar(d['day'] + 1, d['minutes'], color=[colors[c] for c in d['condition']])
for c in d['condition'].unique():
    ax.bar([], [], color=colors[c], label=names[c])
ax.set_xlabel('Study day')
ax.set_ylabel('Minutes scrolling')
ax.set_title('Daily scrolling time')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / '1_daily_minutes.png', dpi=200)
plt.show()

In [ ]:
if 'trig' in globals() and trig['type'].nunique() == 2:
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    vals = trig.groupby('type')['stopped_3min'].mean().reindex(['shadow_nudge', 'nudge']) * 100
    ax.bar(['No message\n(shadow nudge)', 'Nudge shown'], vals, color=['#2F6DB5', '#3A9D5D'])
    for i, v in enumerate(vals):
        ax.text(i, v + 1, f'{v:.0f}%', ha='center')
    ax.set_ylabel('Stopped within 3 minutes (%)')
    ax.set_ylim(0, 100)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG / '2_stopping_after_nudge.png', dpi=200)
    plt.show()

In [ ]:
if len(labels):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.scatter(labels['elapsed_min'], labels['rating'] + np.random.default_rng(1).uniform(-0.15, 0.15, len(labels)),
               color='#2F6DB5', alpha=0.6)
    ax.set_xlabel('Minutes into the session')
    ax.set_ylabel('Intentionality (1 = mindless)')
    ax.set_title('Does scrolling become less intentional over time?')
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG / '3_intentionality_vs_time.png', dpi=200)
    plt.show()

## Reading the results

- This is a single-person study, so it shows what works for one person. It is a pilot, not a general finding.
- With 7 nudge and 7 no-nudge days, only fairly large effects can reach statistical significance. Report effect sizes along with p-values.
- Knowing that you are in an experiment may change how you scroll, especially in the first days.